# Figure 1A: CP3 per-site ¹H shift spread vs the DFT and experimental resolution limits

Histogram of per-site ¹H shift spread across the Goodman CP3 stereoisomers (Goodman, JOC 2009, 74,
4597), against the experimental noise floor (0.02 ppm) and DFT's resolving power (0.10 ppm).

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("analysis/code",):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FixedLocator

import cp3

In [ ]:
def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
variations = cp3.load_variations(nucleus="H")
frac = cp3.fraction_below_dft(variations)
print(f"{len(variations)} proton sites; {frac:.1%} fall between the experimental floor "
      f"({cp3.EXPERIMENTAL_LIMIT} ppm) and the DFT limit ({cp3.DFT_LIMIT} ppm)")

In [ ]:
# color per zone; keys match cp3._zone's return values
zone_colors = {
    "below_experimental": "#2c4553",  # < 0.02 ppm: within experimental noise
    "below_dft": "#a72608",           # 0.02 - 0.10 ppm: informative but DFT cannot resolve
    "dft_zone": "#dacd82",            # 0.10 - 0.30 ppm: resolvable by DFT
    "large": "#68a79a",               # >= 0.30 ppm
}
boundaries = (cp3.EXPERIMENTAL_LIMIT, cp3.DFT_LIMIT, cp3.LARGE_VARIATION)

In [ ]:
sns.set_theme(context="paper", style="white")
fig, ax = plt.subplots(figsize=(8, 4))

bins = 30
edges = np.linspace(0.0, float(variations.max()), bins + 1)
counts, edge = np.histogram(variations, bins=edges)

# Draw each bar, splitting any bin that straddles a zone boundary so each segment gets its color.
for height, left, right in zip(counts, edge[:-1], edge[1:]):
    splits = sorted({left, right} | {b for b in boundaries if left < b < right})
    for a, b in zip(splits[:-1], splits[1:]):
        ax.bar(a, height, width=b - a, align="edge",
               color=zone_colors[cp3._zone(0.5 * (a + b))],
               edgecolor="white", linewidth=0.6, alpha=0.9)

ax.set_xlim(left=0.0)
ax.set_ylim(0, ax.get_ylim()[1])
ax.margins(x=0)
ax.set_title("Chemical shift variation (¹H)", pad=8, fontsize=16)
ax.set_xlabel("Variation at Site (standard deviation, ¹H ppm)", fontsize=13)
ax.set_ylabel("Count", fontsize=13)
sns.despine(ax=ax, top=True, right=True)
for side in ("left", "bottom"):
    ax.spines[side].set_linewidth(0.6)
ax.xaxis.set_major_locator(FixedLocator([0.0, 0.1, 0.2, 0.3, 0.4, 0.5]))
ax.tick_params(axis="both", which="major", labelsize=10, size=1.5, width=0.6)
ax.minorticks_off()
fig.tight_layout()

fig.savefig(figure_path("fig1a_cp3_1H.png"), dpi=300, bbox_inches="tight", pad_inches=0.02)
plt.show()